# Pose-Booth RTX 3060 local validation

Run this notebook with the `apps/api/.venv` kernel after the native API environment is installed. It records observed values only; it contains no assumed performance numbers.

In [1]:
import json, os, platform, statistics, subprocess, sys, time, urllib.request
from pathlib import Path

API_DIR = (Path.cwd().parent / 'apps' / 'api').resolve() if Path.cwd().name == 'notebooks' else (Path.cwd() / 'apps' / 'api').resolve()
sys.path.insert(0, str(API_DIR))
os.environ.setdefault('AI_PROFILE', 'studio')
os.environ.setdefault('DEVICE', 'cuda:0')
os.environ.setdefault('USE_FP16', 'true')
print({'python': sys.version, 'platform': platform.platform(), 'api_dir': str(API_DIR)})

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'api_dir': 'C:\\FLM\\Ki7\\EXE101\\pose-booth-ai\\apps\\api'}


In [2]:
import torch, torchvision, ultralytics
assert torch.cuda.is_available(), 'CUDA is not available to PyTorch'
device = torch.cuda.get_device_properties(0)
runtime = {
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'ultralytics': ultralytics.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': device.name,
    'vram_gib': round(device.total_memory / 1024**3, 2),
}
runtime

{'torch': '2.11.0+cu128',
 'torchvision': '0.26.0+cu128',
 'ultralytics': '8.4.162',
 'cuda_runtime': '12.8',
 'gpu': 'NVIDIA GeForce RTX 3060',
 'vram_gib': 12.0}

In [3]:
from core.engine import YOLOv8PoseEngine
import numpy as np

torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
engine = YOLOv8PoseEngine()
load_ms = (time.perf_counter() - started) * 1000
assert engine.is_ready(), engine.load_error
sample = np.zeros((engine.img_size, engine.img_size, 3), dtype=np.uint8)
latencies = []
for _ in range(12):
    tick = time.perf_counter()
    engine.predict(sample)
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - tick) * 1000)
usable = latencies[2:]
benchmark = {
    'profile': engine.profile, 'device': engine.device, 'fp16': engine.use_fp16,
    'img_size': engine.img_size, 'load_ms': round(load_ms, 2),
    'mean_latency_ms': round(statistics.mean(usable), 2),
    'p95_latency_ms': round(sorted(usable)[int(len(usable) * 0.95) - 1], 2),
    'throughput_fps': round(1000 / statistics.mean(usable), 2),
    'peak_vram_mib': round(torch.cuda.max_memory_allocated() / 1024**2, 2),
}
benchmark

WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


...

WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


{'profile': 'studio',
 'device': 'cuda:0',
 'fp16': True,
 'img_size': 640,
 'load_ms': 16712.18,
 'mean_latency_ms': 28.47,
 'p95_latency_ms': 29.23,
 'throughput_fps': 35.13,
 'peak_vram_mib': 330.55}

In [4]:
# Start the API separately before this cell: uvicorn main:app --host 127.0.0.1 --port 8000
with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=10) as response:
    health = json.load(response)
assert health['profile'] == 'studio'
assert health['device'] == 'cuda:0'
assert health['fp16_enabled'] is True
assert health['engine_ready'] is True
health

{'status': 'ok',
 'profile': 'studio',
 'device': 'cuda:0',
 'fp16_enabled': True,
 'engine_ready': True,
 'library_poses_cached': 20}